# Cassandra: le interazioni principali

Questo notebook gira nel container Jupyter del laboratorio e si connette al cluster
Cassandra tramite le variabili d'ambiente definite nel `docker-compose.yml`
(`CASSANDRA_HOSTS`, `REPLICATION_FACTOR`), quindi **funziona identico sul laboratorio
single-node e su quello a tre nodi**.

Indice: connessione · keyspace · tabelle e chiavi · CRUD · prepared statement e batch ·
TTL e LWT · collezioni e UDT · modellazione per query · consistency level ·
replica e token · pandas · pulizia.

## 1. Connessione

In [ ]:
import os, uuid, datetime as dt
from cassandra.cluster import Cluster, ExecutionProfile, EXEC_PROFILE_DEFAULT
from cassandra import ConsistencyLevel
from cassandra.query import SimpleStatement, BatchStatement
from cassandra.policies import DCAwareRoundRobinPolicy, TokenAwarePolicy

HOSTS = os.environ.get("CASSANDRA_HOSTS", "cassandra").split(",")
PORT = int(os.environ.get("CASSANDRA_PORT", "9042"))
RF = int(os.environ.get("REPLICATION_FACTOR", "1"))

profile = ExecutionProfile(
    load_balancing_policy=TokenAwarePolicy(DCAwareRoundRobinPolicy(local_dc="datacenter1")),
)
cluster = Cluster(HOSTS, port=PORT, execution_profiles={EXEC_PROFILE_DEFAULT: profile})
session = cluster.connect()

print("Cluster:", cluster.metadata.cluster_name)
for h in cluster.metadata.all_hosts():
    print(f"  {h.address}  dc={h.datacenter} rack={h.rack} up={h.is_up}")
print("Replication factor del laboratorio:", RF)

Gli stessi dati si leggono anche via CQL dalle tabelle di sistema. Con tre nodi
`system.peers` mostra gli altri due.

In [ ]:
print(session.execute("SELECT release_version, cluster_name, data_center FROM system.local").one())
for r in session.execute("SELECT peer, data_center, rack FROM system.peers"):
    print(r)

## 2. Keyspace

Il keyspace definisce la strategia di replica. `NetworkTopologyStrategy` è quella da
usare anche in laboratorio (la `SimpleStrategy` ignora datacenter e rack).

In [ ]:
session.execute(f"""
CREATE KEYSPACE IF NOT EXISTS lab
WITH replication = {{'class': 'NetworkTopologyStrategy', 'datacenter1': {RF}}}
""")
session.set_keyspace("lab")
print(session.execute("SELECT keyspace_name, replication FROM system_schema.keyspaces WHERE keyspace_name='lab'").one())

## 3. Tabelle e chiavi primarie

`PRIMARY KEY ((partition_key), clustering_columns)`:

* la **partition key** decide *su quali nodi* finisce la riga (hash → token);
* le **clustering column** decidono *l'ordine* delle righe dentro la partizione.

Qui: una partizione per sensore, e le letture ordinate dalla più recente.

In [ ]:
session.execute("""
CREATE TABLE IF NOT EXISTS readings (
    sensor_id  text,
    ts         timestamp,
    temperature double,
    humidity    double,
    PRIMARY KEY ((sensor_id), ts)
) WITH CLUSTERING ORDER BY (ts DESC)
""")
print("Tabella readings pronta")

## 4. Inserimento: prepared statement e batch

In [ ]:
insert = session.prepare(
    "INSERT INTO readings (sensor_id, ts, temperature, humidity) VALUES (?, ?, ?, ?)"
)

import random
random.seed(42)
now = dt.datetime.now(dt.timezone.utc).replace(microsecond=0)
for sensor in ("s1", "s2", "s3"):
    for i in range(20):
        session.execute(insert, (
            sensor, now - dt.timedelta(minutes=i),
            round(random.uniform(18, 28), 1), round(random.uniform(30, 70), 1),
        ))
print("Inseriti 60 record")

Un **batch** è utile per scrivere più righe della *stessa partizione* in modo atomico.
Non è un meccanismo di performance: batch su partizioni diverse sono più lenti.

In [ ]:
batch = BatchStatement()
for i in range(20, 25):
    batch.add(insert, ("s1", now - dt.timedelta(minutes=i), 20.0, 50.0))
session.execute(batch)
print("Batch eseguito")

## 5. Lettura

In [ ]:
# Tutta una partizione (già ordinata per ts DESC)
for r in session.execute("SELECT * FROM readings WHERE sensor_id='s1' LIMIT 3"):
    print(r)

In [ ]:
# Range sulla clustering column
since = now - dt.timedelta(minutes=5)
rows = session.execute(
    "SELECT ts, temperature FROM readings WHERE sensor_id=%s AND ts >= %s", ("s2", since)
)
print([ (r.ts.strftime("%H:%M"), r.temperature) for r in rows ])

In [ ]:
# Aggregazioni dentro una partizione
print(session.execute(
    "SELECT count(*) AS n, avg(temperature) AS avg_t, max(temperature) AS max_t "
    "FROM readings WHERE sensor_id='s3'").one())

Cassandra **non** permette di filtrare liberamente: una query senza partition key o su
colonne non indicizzate viene rifiutata (a meno di `ALLOW FILTERING`, da evitare).

In [ ]:
try:
    session.execute("SELECT * FROM readings WHERE temperature > 25")
except Exception as e:
    print(type(e).__name__, "->", str(e)[:200])

## 6. Update, delete, TTL

In [ ]:
ts0 = now
session.execute("UPDATE readings SET temperature=%s WHERE sensor_id=%s AND ts=%s", (99.9, "s1", ts0))
print(session.execute("SELECT temperature FROM readings WHERE sensor_id='s1' AND ts=%s", (ts0,)).one())

session.execute("DELETE FROM readings WHERE sensor_id=%s AND ts=%s", ("s1", ts0))
print("Dopo delete:", session.execute("SELECT * FROM readings WHERE sensor_id='s1' AND ts=%s", (ts0,)).one())

In [ ]:
# TTL: la riga sparisce da sola dopo N secondi
session.execute(
    "INSERT INTO readings (sensor_id, ts, temperature, humidity) VALUES ('tmp', toTimestamp(now()), 1, 1) USING TTL 30"
)
print(session.execute("SELECT sensor_id, TTL(temperature) AS ttl FROM readings WHERE sensor_id='tmp'").one())

## 7. Lightweight transactions (compare-and-set)

`IF NOT EXISTS` / `IF ...` usano Paxos: servono quando serve davvero unicità o
concorrenza controllata, ma costano più round trip.

In [ ]:
session.execute("""
CREATE TABLE IF NOT EXISTS users (
    username text PRIMARY KEY,
    email text,
    tags set<text>,
    phones list<text>,
    prefs map<text, text>
)
""")
name = "alice_" + uuid.uuid4().hex[:6]   # nome nuovo a ogni esecuzione: la cella è rieseguibile
q = "INSERT INTO users (username, email) VALUES (%s, %s) IF NOT EXISTS"
r1 = session.execute(q, (name, "alice@example.com")).one()
r2 = session.execute(q, (name, "altra@example.com")).one()
print("Primo insert applicato: ", r1.applied)
print("Secondo insert applicato:", r2.applied, "| valore esistente:", r2.email)

## 8. Collezioni (set, list, map) e tipi definiti dall'utente

In [ ]:
session.execute("UPDATE users SET tags = tags + {'admin','dev'} WHERE username=%s", (name,))
session.execute("UPDATE users SET phones = ['+39 06 1234567'] WHERE username=%s", (name,))
session.execute("UPDATE users SET prefs['lang'] = 'it', prefs['theme'] = 'dark' WHERE username=%s", (name,))
print(session.execute("SELECT * FROM users WHERE username=%s", (name,)).one())

In [ ]:
session.execute("CREATE TYPE IF NOT EXISTS address (street text, city text, zip text)")
session.execute("CREATE TABLE IF NOT EXISTS customers (id uuid PRIMARY KEY, name text, addr frozen<address>)")
cid = uuid.uuid4()
session.execute(
    "INSERT INTO customers (id, name, addr) VALUES (%s, 'Bob', {street:'Via Roma 1', city:'Roma', zip:'00100'})", (cid,))
print(session.execute("SELECT name, addr.city FROM customers WHERE id=%s", (cid,)).one())

## 9. Modellare per le query (denormalizzazione)

In Cassandra si progetta **una tabella per ogni query**. Vogliamo anche cercare
gli utenti per email: non si fa con un join o con un filtro, ma con una seconda tabella.

In [ ]:
session.execute("""
CREATE TABLE IF NOT EXISTS users_by_email (email text PRIMARY KEY, username text)
""")
b = BatchStatement()
b.add(SimpleStatement("INSERT INTO users (username, email) VALUES (%s, %s)"), ("carol", "carol@example.com"))
b.add(SimpleStatement("INSERT INTO users_by_email (email, username) VALUES (%s, %s)"), ("carol@example.com", "carol"))
session.execute(b)   # logged batch: entrambe le scritture o nessuna
print(session.execute("SELECT * FROM users_by_email WHERE email='carol@example.com'").one())

Un indice secondario (`CREATE INDEX`) esiste, ma è adatto solo a colonne a bassa cardinalità
e interroga tutti i nodi: usalo con cautela.

In [ ]:
session.execute("CREATE INDEX IF NOT EXISTS users_email_idx ON users (email)")
import time; time.sleep(2)
print(session.execute("SELECT username FROM users WHERE email='alice@example.com' LIMIT 1").one())

## 10. Consistency level

Ogni operazione sceglie quanti replica devono rispondere. Con RF=3 (laboratorio a tre
nodi): `ONE` = 1, `QUORUM` = 2, `ALL` = 3. Regola d'oro: **R + W > RF** ⇒ letture
consistenti (es. QUORUM + QUORUM).

In [ ]:
def run(cl, cql, params=None):
    stmt = SimpleStatement(cql, consistency_level=cl)
    return session.execute(stmt, params)

run(ConsistencyLevel.QUORUM,
    "INSERT INTO users (username, email) VALUES ('dave','dave@example.com')")
for cl in (ConsistencyLevel.ONE, ConsistencyLevel.QUORUM, ConsistencyLevel.ALL):
    try:
        row = run(cl, "SELECT username FROM users WHERE username='dave'").one()
        print(f"{ConsistencyLevel.value_to_name[cl]:7} -> {row}")
    except Exception as e:
        print(f"{ConsistencyLevel.value_to_name[cl]:7} -> {type(e).__name__}")

> **Esperimento (solo tre nodi).** Da un terminale sul tuo computer esegui
> `docker stop cassandra3` e rilancia la cella: `ONE` e `QUORUM` funzionano ancora,
> `ALL` fallisce con `Unavailable`. Poi `docker start cassandra3`.

## 11. Replica e token

Il driver conosce l'anello dei token: possiamo vedere quali nodi ospitano una
partizione. Con RF=1 c'è un solo nodo; con RF=3 tutti e tre.

In [ ]:
from cassandra.metadata import Murmur3Token
for sensor in ("s1", "s2", "s3"):
    key = session.prepare("SELECT * FROM readings WHERE sensor_id=?").bind((sensor,)).routing_key
    replicas = cluster.metadata.get_replicas("lab", key)
    print(sensor, "->", [str(h.address) for h in replicas])

print()
for r in session.execute("SELECT sensor_id, token(sensor_id) AS tok FROM readings PER PARTITION LIMIT 1"):
    print(r.sensor_id, r.tok)

## 12. Integrazione con pandas

In [ ]:
import pandas as pd

rows = session.execute("SELECT * FROM readings WHERE sensor_id='s1' LIMIT 200")
df = pd.DataFrame(list(rows))   # le Row sono namedtuple: pandas ricava le colonne
display(df.head())
print(df[["temperature", "humidity"]].describe())

## 13. Pulizia

Rimuove il keyspace del laboratorio e chiude la connessione.

In [ ]:
# session.execute("DROP KEYSPACE IF EXISTS lab")   # <- decommenta per ripulire tutto
cluster.shutdown()
print("Connessione chiusa")